<a href="https://colab.research.google.com/github/swapnasaina274/malicious-mobile-app-detection/blob/main/apk_extractor_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install androguard langchain

import os
from google.colab import drive

# Mount Google Drive to access the uploaded APK file
drive.mount('/content/drive')


import os
import shutil
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Define the path where the files are stored in Drive
drive_path = "/content/drive/MyDrive/Malicious project work"

# 3. Copy all .pkl files into local Colab storage (/content/)
pkl_files = ["champion_lightgbm_model.pkl", "class_mapping.pkl", "feature_names.pkl", "scaler.pkl"]

for file in pkl_files:
    src = os.path.join(drive_path, file)
    dst = os.path.join("/content", file)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✅ Loaded {file} into local workspace")
    else:
        print(f"⚠️ Warning: {file} not found at {src}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# APK FEATURE EXTRACTION & LANGCHAIN TOOL INTEGRATION


import os
import joblib
import numpy as np
import pandas as pd
from langchain.tools import tool

# Check if AndroGuard is available for static analysis
try:
    from androguard.misc import AnalyzeAPK
    ANDROGUARD_AVAILABLE = True
except ImportError:
    ANDROGUARD_AVAILABLE = False


def extract_features_from_apk(apk_path: str, feature_names_path: str = "feature_names.pkl") -> pd.DataFrame:
    """
    Parses an APK using AndroGuard to extract system call frequencies
    and constructs a 1 x 470 vector matching the training feature set.
    """
    if not ANDROGUARD_AVAILABLE:
        raise ImportError("AndroGuard is not installed. Run '!pip install androguard' to enable static analysis.")

    print(f"[+] Analyzing APK: {apk_path}...")

    # Load 470 feature column names saved during Step 9
    feature_names = joblib.load(feature_names_path)

    # Decompile DEX and set up analysis
    a, d, dx = AnalyzeAPK(apk_path)

    # Initialize 1x470 DataFrame with zeros
    feature_vector = pd.DataFrame(0, index=[0], columns=feature_names)

    # Count matching method calls across DEX files
    for method in dx.get_methods():
        method_name = method.name.lower()
        if method_name in feature_vector.columns:
            feature_vector.at[0, method_name] += 1

    print("[+] Feature extraction complete!")
    return feature_vector


@tool
def analyze_android_apk(apk_file_path: str) -> str:
    """
    Analyzes an Android APK file for malware threats.
    Input: Path to the .apk file on disk.
    Output: Comprehensive risk report with classification category and prediction confidence score.
    """
    if not os.path.exists(apk_file_path):
        return f"Error: APK file not found at path '{apk_file_path}'."

    try:
        # Load production model artifacts
        model = joblib.load("champion_lightgbm_model.pkl")
        class_map = joblib.load("class_mapping.pkl")

        # Step 1: Feature Extraction
        feature_vector = extract_features_from_apk(apk_file_path)

        # Step 2: Prediction
        pred_class_id = model.predict(feature_vector)[0]
        predicted_label = class_map[pred_class_id]

        probabilities = model.predict_proba(feature_vector)[0]
        confidence = round(float(np.max(probabilities)) * 100, 2)

        # Step 3: Format natural language report for LangChain Agent reasoning
        is_malicious = predicted_label != "Benign"
        status_flag = "🚨 THREAT DETECTED" if is_malicious else "✅ CLEAN / SAFE"

        report = (
            f"=== Android Malware Analysis Report ===\n"
            f"Target File : {os.path.basename(apk_file_path)}\n"
            f"Status      : {status_flag}\n"
            f"Category    : {predicted_label} (Class ID: {pred_class_id})\n"
            f"Confidence  : {confidence}%\n\n"
            f"Recommended Action:\n"
            f"{'Quarantine/Remove this application immediately to prevent malicious execution.' if is_malicious else 'The application exhibits standard benign system call frequencies and appears safe to run.'}"
        )
        return report

    except Exception as e:
        return f"An error occurred during APK analysis: {str(e)}"

In [ ]:
from google.colab import files
import os

# 1. Trigger the browser file upload prompt
print("Select com.iris.gallery_16.apk from your computer:")
uploaded = files.upload()

# 2. Get the uploaded file path
if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    apk_path = os.path.join("/content", uploaded_filename)
    print(f"\n[+] File uploaded successfully to: {apk_path}")
else:
    print("\n[-] No file was selected.")

Select com.iris.gallery_16.apk from your computer:


Saving com.iris.gallery_16.apk to com.iris.gallery_16 (1).apk

[+] File uploaded successfully to: /content/com.iris.gallery_16 (1).apk


In [ ]:
# Run inference test on the uploaded file path
test_output = analyze_android_apk.invoke({"apk_file_path": apk_path})
print(test_output)